In [ ]:
class MancalaBoard:
  def __init__(self):

      self.board = [
          [4, 4, 4, 4, 4, 4, 0],
          [4, 4, 4, 4, 4, 4, 0],
      ]
      self.current_player = 0

  def copy(self):
      new_board = MancalaBoard()
      new_board.board = [row[:] for row in self.board]
      new_board.current_player = self.current_player
      return new_board

  def get_legal_moves(self, player):
  # This will return pits that aren't empty (indices 0 to 5)
    return [i for i in range(6) if self.board[player][i] > 0]


  def is_terminal(self):
  # The Game will end when either player's pits will be all empty
    return (all(self.board[0][i] == 0 for i in range(6)) and
          all(self.board[1][i] == 0 for i in range(6)))

  def collect_remaining(self):
      for player in range(2):
          for pit in range(6):
              self.board[player][6] += self.board[player][pit]
              self.board[player][pit] = 0

  def distribute_stones(self, player, pit):
    stones = self.board[player][pit]
    self.board[player][pit] = 0  # Clear the starting pit
    current_side = player
    current_pit_idx = pit
    last_pit_landed = (player, pit) # Track where the last stone lands

    bonus_turn = False

    while stones > 0:
        current_pit_idx += 1

        # Check if we've gone past the last pit on the current side
        # Pit 6 is the store. Pit 0-5 are regular pits.
        if current_pit_idx > 6: # Past the store on the current side
            current_side = 1 - current_side # Switch to the opponent's side
            current_pit_idx = 0 # Start from the first pit on the new side

        # Skip opponent's store
        if current_side != player and current_pit_idx == 6:
            continue # Skip and go to next pit

        # Deposit a stone
        self.board[current_side][current_pit_idx] += 1
        stones -= 1
        last_pit_landed = (current_side, current_pit_idx)

    # After all stones are distributed, check for bonus turn and capture rule
    # Bonus turn: last stone landed in current player's store
    if last_pit_landed[0] == player and last_pit_landed[1] == 6:
        bonus_turn = True

    # Capture rule: last stone landed in an empty pit on current player's side (not store)
    # and the opposite pit has stones.
    elif last_pit_landed[0] == player and last_pit_landed[1] != 6 and self.board[player][last_pit_landed[1]] == 1:
        opposite_pit_idx = 5 - last_pit_landed[1]
        captured_stones = self.board[1 - player][opposite_pit_idx]

        if captured_stones > 0:
            self.board[player][6] += captured_stones + 1 # +1 for the stone just placed
            self.board[1 - player][opposite_pit_idx] = 0
            self.board[player][last_pit_landed[1]] = 0 # Clear the pit on player's side too

    return bonus_turn

  def make_move(self, pit):
      player = self.current_player
      bonus_turn = self.distribute_stones(player, pit)
      if not bonus_turn:
          self.current_player = 1 - player
      return self

  def __str__(self):
      # Player 1 (AI) board pits (reversed for display) and store
      s = "  P1 (AI) -> " + " ".join([str(x) for x in self.board[1][5::-1]]) + "\n"
      s += f"P1 Store[{self.board[1][6]}] <------------------------> P0 Store[{self.board[0][6]}]\n"
      # Player 0 (Human) board pits and store
      s += "  P0 (You)-> " + " ".join([str(x) for x in self.board[0][0:6]]) + "\n"
      return s

In [ ]:
game = MancalaBoard()
print(game.get_legal_moves(0))
print(game.is_terminal())

[0, 1, 2, 3, 4, 5]
False


In [ ]:
# The definitions for evaluate, alpha_beta, and minimax_agent have been moved to the play_vs_ai cell for better execution context.
# This cell can now be cleared or removed if desired.

In [ ]:
game = MancalaBoard()
move = minimax_agent(game, depth=4)
print(f"Best move for Player 1:Pit {move}") # This should print a number 0-5

Best move for Player 1:Pit 2


In [ ]:
def H1(board, player):
    """Hoard seeds in the leftmost pit (furthest from own store)."""
    return board.board[player][0]

def H4(board, player):
    """Number of seeds in own store."""
    return board.board[player][6]

def H6(board, player):
    """Negative of opponent's store; we want this small."""
    return -board.board[1 - player][6]

def H7(board, player):
    """1 if it is still player's turn at this state (last move earned a bonus turn)."""
    return 1 if board.current_player == player else 0

# Weights derived from the average values in Hunter (2021) Table 3.
W1, W4, W6, W7 = 0.2, 1.0, 0.6, 0.9

def evaluate_heuristic(board, player):
    return (W1 * H1(board, player)
          + W4 * H4(board, player)
          + W6 * H6(board, player)
          + W7 * H7(board, player))

In [ ]:
import time

# Heuristic snapshot on the starting board
b = MancalaBoard()
print(f"Start position H1={H1(b,0)}, H4={H4(b,0)}, H6={H6(b,0)}, H7={H7(b,0)}, "
      f"weighted={evaluate_heuristic(b,0):.2f}")

# Both evaluators on a fresh board, depth 6
t0 = time.time()
m1 = minimax_agent(MancalaBoard(), depth=6)
print(f"Plain alpha-beta picks pit {m1} in {time.time()-t0:.2f}s")

t0 = time.time()
m2 = minimax_agent(MancalaBoard(), depth=6)
print(f"Heuristic minimax picks pit {m2} in {time.time()-t0:.2f}s")

# One full game: heuristic (P0) vs plain (P1) at depth 4
g = MancalaBoard()
while not g.is_terminal():
    fn = evaluate_heuristic if g.current_player == 0 else evaluate
    move = minimax_agent(g, depth=4)
    if move is None:
        break
    g.make_move(move)
g.collect_remaining()
print(f"Final score - P0 heuristic: {g.board[0][6]}, P1 plain: {g.board[1][6]}")

Start position H1=4, H4=0, H6=0, H7=1, weighted=1.70
Plain alpha-beta picks pit 2 in 0.28s
Heuristic minimax picks pit 2 in 0.16s
Final score - P0 heuristic: 33, P1 plain: 15


In [ ]:
from IPython.display import clear_output
g = MancalaBoard()
print("=== Starting position ===")
print(g)

move_num = 0
while not g.is_terminal():
    fn = evaluate_heuristic if g.current_player == 0 else evaluate
    move = minimax_agent(g, depth=4)
    if move is None:
        break
    move_num += 1
    print(f"\n--- Move {move_num}: P{g.current_player} plays pit {move} ---")
    g.make_move(move)
    print(g)

g.collect_remaining()
print(f"\nFinal score: P0 (heuristic) = {g.board[0][6]}, P1 (plain) = {g.board[1][6]}")

=== Starting position ===
  P1 (AI) -> 4 4 4 4 4 4
P1 Store[0] <------------------------> P0 Store[0]
  P0 (You)-> 4 4 4 4 4 4


--- Move 1: P0 plays pit 2 ---
  P1 (AI) -> 4 4 4 4 4 4
P1 Store[0] <------------------------> P0 Store[1]
  P0 (You)-> 4 4 0 5 5 5


--- Move 2: P0 plays pit 5 ---
  P1 (AI) -> 4 4 5 5 5 5
P1 Store[0] <------------------------> P0 Store[2]
  P0 (You)-> 4 4 0 5 5 0


--- Move 3: P1 plays pit 1 ---
  P1 (AI) -> 5 5 6 6 0 5
P1 Store[1] <------------------------> P0 Store[2]
  P0 (You)-> 4 4 0 5 5 0


--- Move 4: P1 plays pit 0 ---
  P1 (AI) -> 6 6 7 7 1 0
P1 Store[1] <------------------------> P0 Store[2]
  P0 (You)-> 4 4 0 5 5 0


--- Move 5: P0 plays pit 4 ---
  P1 (AI) -> 6 6 7 8 2 1
P1 Store[1] <------------------------> P0 Store[3]
  P0 (You)-> 4 4 0 5 0 1


--- Move 6: P1 plays pit 1 ---
  P1 (AI) -> 6 6 8 9 0 1
P1 Store[1] <------------------------> P0 Store[3]
  P0 (You)-> 4 4 0 5 0 1


--- Move 7: P0 plays pit 5 ---
  P1 (AI) -> 6 6 8 9 0 1
P1 Store[1]

In [ ]:
import random

def random_agent(board):
  moves = board.get_legal_moves(board.current_player)
  if not moves: # Check if the list of moves is empty
    return None # No legal moves available
  return random.choice(moves)

def play_game():
  board = MancalaBoard()

  while not board.is_terminal():
    move = random_agent(board)
    if move is None: # If no legal moves for current player
      break # End the game loop
    board.make_move(move)

  board.collect_remaining()
  p1 = board.board[0][6]
  p2= board.board[1][6]

  print(f"Player 1: {p1} stones")
  print(f"Player 2: {p2} stones")

  if p1 > p2:
    print("Player 1 Wins!")
  elif p2 > p1:
    print("Player 2 Wins!")
  else:
    print("Draw!")

# Run it
play_game()

Player 1: 17 stones
Player 2: 31 stones
Player 2 Wins!


In [ ]:
def print_board(board):
    print(board)

def human_move(board):
    player = board.current_player
    while True:
        try:
            prompt = f"Player {player + 1} (You), enter your move (pit 1-6): "
            move = int(input(prompt)) - 1  # Convert to 0-indexed
            if move in board.get_legal_moves(player):
                return move
            else:
                print("Invalid move. Choose an available pit with stones.")
        except ValueError:
            print("Invalid input. Please enter a number.")

#This is the Alpha-Beta Minimax which will be the core of the project
def evaluate(board,player):
  #This will be store the differential H4 from the research
  return board.board[player][6] - board.board[1-player][6]

def alpha_beta(board, depth, alpha, beta, player):
    if board.is_terminal() or depth == 0:
        if board.is_terminal():
            board.collect_remaining()
        return evaluate(board,player)

    current = board.current_player

    if current == player: # This is Maximizing
        best_value = float('-inf')
        for move in board.get_legal_moves(current):
            new_board = board.copy()
            new_board.make_move(move)

            new_depth = depth if new_board.current_player == current else depth -1
            value = alpha_beta(new_board, new_depth, alpha, beta, player)
            best_value = max(best_value, value)
            alpha = max(alpha, best_value)
            if beta <= alpha:
              break # We will prune now
        return best_value

    else: # This is the minimizing part
        best_value = float('inf')
        for move in board.get_legal_moves(current):
            new_board = board.copy()
            new_board.make_move(move)
            new_depth = depth if new_board.current_player == current else depth -1
            value= alpha_beta(new_board, new_depth, alpha, beta, player)
            best_value = min(best_value, value)
            beta = min(beta, best_value)
            if beta <= alpha:
                  break # Prune
        return best_value

def minimax_agent(board, depth=8):
    player = board.current_player
    legal_moves = board.get_legal_moves(player)

    if not legal_moves:
        # If there are no legal moves, AI cannot choose any.
        # This case should ideally be caught by play_vs_ai before calling minimax_agent,
        # but for robustness, minimax_agent should explicitly return None here.
        return None

    best_move = None
    best_value = float('-inf')

    for move in legal_moves:
        new_board = board.copy()
        new_board.make_move(move)

        value = alpha_beta(new_board, depth -1, float('-inf'), float('inf'), player)
        if value > best_value:
              best_value = value
              best_move = move

    # Fallback: if for some reason best_move is still None (theoretically impossible if legal_moves is not empty
    # and alpha_beta returns valid numbers), return the first legal move to prevent errors.
    if best_move is None and legal_moves:
        return legal_moves[0]

    return best_move

def play_vs_ai():
    board = MancalaBoard()
    print("You are Player 1. AI is Player 2\n")

    while True:
        print_board(board)

        # Check for legal moves for the current player
        current_player_legal_moves = board.get_legal_moves(board.current_player)

        if not current_player_legal_moves:
            print(f"Player {board.current_player + 1} has no legal moves. Game ending.")
            break # Current player has no moves, end the game

        move = None
        if board.current_player == 0:
            move = human_move(board)        # Human turn, this function ensures a legal move is returned
        else:
            print("AI is thinking...")
            move = minimax_agent(board, depth=6) # AI will return a legal move or None if an issue occurs

        # IMPORTANT: Check if move is None immediately after determining it
        if move is None:
            print(f"Error: No valid move determined for player {board.current_player + 1}. Aborting game.")
            break

        # Only print AI's choice if it was the AI's turn
        if board.current_player != 0:
            print(f"AI chose pit {move + 1}")

        board.make_move(move)

        # After making a move, check if the game has become terminal
        if board.is_terminal():
            print("Game is terminal after a move.")
            break

    board.collect_remaining()
    p1 = board.board[0][6]
    p2 = board.board[1][6]

    print_board(board) # Print final board state
    print(f"Final score: Player 1 = {p1} stones, Player 2 = {p2} stones")
    if p1 > p2:
        print("Player 1 Wins!")
    elif p2 > p1:
        print("Player 2 Wins!")
    else:
        print("Draw!")

play_vs_ai()

You are Player 1. AI is Player 2

  P1 (AI) -> 4 4 4 4 4 4
P1 Store[0] <------------------------> P0 Store[0]
  P0 (You)-> 4 4 4 4 4 4

Player 1 (You), enter your move (pit 1-6): 1
  P1 (AI) -> 4 4 4 4 4 4
P1 Store[0] <------------------------> P0 Store[0]
  P0 (You)-> 0 5 5 5 5 4

AI is thinking...
AI chose pit 3
  P1 (AI) -> 5 5 5 0 4 4
P1 Store[1] <------------------------> P0 Store[0]
  P0 (You)-> 0 5 5 5 5 4

AI is thinking...
AI chose pit 6
  P1 (AI) -> 0 5 5 0 4 4
P1 Store[2] <------------------------> P0 Store[0]
  P0 (You)-> 1 6 6 6 5 4

Player 1 (You), enter your move (pit 1-6): 3
  P1 (AI) -> 0 5 5 0 5 5
P1 Store[2] <------------------------> P0 Store[1]
  P0 (You)-> 1 6 0 7 6 5

AI is thinking...
AI chose pit 2
  P1 (AI) -> 1 6 6 1 0 5
P1 Store[3] <------------------------> P0 Store[1]
  P0 (You)-> 1 6 0 7 6 5

AI is thinking...
AI chose pit 6
  P1 (AI) -> 0 6 6 1 0 5
P1 Store[4] <------------------------> P0 Store[1]
  P0 (You)-> 1 6 0 7 6 5

AI is thinking...
AI chose pit